<a href="https://colab.research.google.com/github/RakinduNiwunhella/RiceVision/blob/Yield_Model/YieldModle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install geopandas xgboost shapely pyproj fiona

Load all monthly satellite data

In [2]:
BASE_PATH = "/content/drive/MyDrive/SDGP Docs/Datasets/500m datasets"

In [3]:
import glob
import pandas as pd

files = glob.glob(f"{BASE_PATH}/**/*.csv", recursive=True)

print("Total CSV files found:", len(files))

Total CSV files found: 1136


Load & combine all CSVs into ONE DataFrame

In [5]:
import pandas as pd
import os

df_list = []

def extract_district(path):
    parts = path.split(os.sep)
    district = parts[-3]   # district folder
    return district

for f in files:
    temp = pd.read_csv(f)
    temp["district"] = extract_district(f)
    df_list.append(temp)

sat_df = pd.concat(df_list, ignore_index=True)

print("Combined shape:", sat_df.shape)
sat_df[["district"]].head()

Combined shape: (9743347, 34)


,district
0,Galle
1,Galle
2,Galle
3,Galle
4,Galle


In [6]:
sat_df["district"].nunique()

24

In [7]:
sorted(sat_df["district"].unique())

['Ampara',
 'Badulla',
 'Batticaloa',
 'Colombo',
 'Galle',
 'Gampaha',
 'Hambantota',
 'Jaffna',
 'Kaluthara',
 'Kandy',
 'Kegalle',
 'Kilinochchi',
 'Kurunegala',
 'Mannar',
 'Matale',
 'Matara',
 'Monaragala',
 'Mullattivu',
 'Nuwara_Eliya',
 'Polonnaruwa',
 'Puttalam',
 'Ratnapura',
 'Trincomalee',
 'Vavuniya']

Convert date and extract year & month

In [8]:
sat_df["date"] = pd.to_datetime(sat_df["date"])
sat_df["year"] = sat_df["date"].dt.year
sat_df["month"] = sat_df["date"].dt.month

In [9]:
sat_df["year"].value_counts().sort_index()

,count
year,
2022,2336026
2023,2361190
2024,2280849
2025,2765282


In [10]:
sat_df = sat_df[
    (sat_df["cloud_pct"] < 50) &
    (sat_df["SCL"].isin([4,5,6]))
]

print("After cloud filtering:", sat_df.shape)

After cloud filtering: (2540273, 36)


Create vegetation indices

In [11]:
sat_df["NDVI"] = (sat_df["B8"] - sat_df["B4"]) / (sat_df["B8"] + sat_df["B4"])
sat_df["NDWI"] = (sat_df["B3"] - sat_df["B8"]) / (sat_df["B3"] + sat_df["B8"])
sat_df["EVI"] = 2.5 * (sat_df["B8"] - sat_df["B4"]) / (
    sat_df["B8"] + 6*sat_df["B4"] - 7.5*sat_df["B2"] + 1
)

sat_df = sat_df[(sat_df["NDVI"] > -1) & (sat_df["NDVI"] < 1)]

 Assign season

In [12]:
def get_season(month):
    return "Maha" if month in [9,10,11,12,1,2,3] else "Yala"

sat_df["season"] = sat_df["month"].apply(get_season)

In [13]:
print("Years:", sat_df["year"].unique())
print("Districts:", sat_df["district"].nunique())
print(sat_df[["district", "year", "season"]].drop_duplicates().head())

Years: [2022 2023 2024 2025]
Districts: 24
      district  year season
550      Galle  2022   Yala
809      Galle  2022   Maha
9894     Galle  2023   Yala
12108    Galle  2023   Maha
18996    Galle  2024   Maha


Aggregate to DISTRICT–YEAR–SEASON

In [14]:
agg_df = sat_df.groupby(
    ["district", "year", "season"]
).agg({
    "NDVI": ["mean", "max", "std"],
    "EVI": ["mean"],
    "NDWI": ["mean"],
    "rain_30d": "mean",
    "rain_7d": "mean",
    "tmean": "mean",
    "tmax": "mean",
    "rh_mean": "mean",
    "elevation": "mean",
    "slope": "mean"
}).reset_index()

agg_df.columns = [
    "_".join(col).strip("_") for col in agg_df.columns
]